# Gale-Roth-Shapley Matching Engine in Python
A production-ready reference implementation of the Gale-Roth-Shapley Deferred Acceptance algorithm for stable 1:1 resource allocation (e.g., student–school admissions, candidate–job placement, medical residency matching).

## Notebook Contents:

- **Matching Solver (gale_roth_shapley)**: Computes stable pairings from dictionary-based preferences and outputs a canonical list of (proposer, receiver) matched tuples.

- **Stability Checker (is_stable)**: Verifies the outcome against blocking-pair conditions and individual rationality.

- **Automated Unit Tests**: Executes comprehensive test suites directly within the IPython/Colab runtime.

## Gale-Roth-Shapley Algorithm

In [24]:
from collections import deque

def validate_preferences(
    proposers_pref: dict[str, list[str]],
    receivers_pref: dict[str, list[str]]
) -> None:
    """
    Validates structural integrity of market preferences.
    Raises ValueError or TypeError if preferences are malformed.
    """
    if not isinstance(proposers_pref, dict) or not isinstance(receivers_pref, dict):
        raise TypeError("Preferences must be dictionaries mapping agent names to preference lists.")

    proposer_set = set(proposers_pref.keys())
    receiver_set = set(receivers_pref.keys())

    # Check proposers
    for proposer, prefs in proposers_pref.items():
        if not isinstance(prefs, list):
            raise TypeError(f"Preference list for proposer '{proposer}' must be a list.")
        if len(prefs) != len(set(prefs)):
            raise ValueError(f"Proposer '{proposer}' contains duplicate preferences: {prefs}")
        # Ensure all listed receivers actually exist in the market
        unknown = set(prefs) - receiver_set
        if unknown:
            raise ValueError(f"Proposer '{proposer}' lists unknown receiver(s): {unknown}")

    # Check receivers
    for receiver, prefs in receivers_pref.items():
        if not isinstance(prefs, list):
            raise TypeError(f"Preference list for receiver '{receiver}' must be a list.")
        if len(prefs) != len(set(prefs)):
            raise ValueError(f"Receiver '{receiver}' contains duplicate preferences: {prefs}")
        # Ensure all listed proposers actually exist in the market
        unknown = set(prefs) - proposer_set
        if unknown:
            raise ValueError(f"Receiver '{receiver}' lists unknown proposer(s): {unknown}")

def gale_roth_shapley(
    proposers_pref: dict[str, list[str]],
    receivers_pref: dict[str, list[str]],
) -> list[tuple[str, str]]:
    """Computes a stable matching between proposers and receivers.

    :param proposers_pref: Dict mapping proposer -> list of receivers in order
      of preference.
    :param receivers_pref: Dict mapping receiver -> list of proposers in order
      of preference.
    :return: List of (proposer, receiver) tuples representing stable pairs.
    """

    # Fail fast if inputs violate strict ordering or refer to unknown entities
    validate_preferences(proposers_pref, receivers_pref)

    # 1. Precompute receiver rank lookups for O(1) preference comparisons
    receiver_ranks = {
        receiver: {proposer: rank for rank, proposer in enumerate(prefs)}
        for receiver, prefs in receivers_pref.items()
    }

    # Track current preference index for each proposer
    proposer_next_idx = {p: 0 for p in proposers_pref}

    # Receivers track their currently held match: {receiver: proposer}
    receiver_matches: dict[str, str] = {}

    # Queue of currently unmatched proposers
    free_proposers = deque(proposers_pref.keys())

    # 2. Main deferred acceptance loop
    while free_proposers:
        proposer = free_proposers.popleft()
        prefs = proposers_pref[proposer]

        # Check if the proposer has exhausted their list
        if proposer_next_idx[proposer] >= len(prefs):
            continue

        # Propose to the highest-ranked receiver not yet approached
        target_receiver = prefs[proposer_next_idx[proposer]]
        proposer_next_idx[proposer] += 1

        # Check if target receiver considers this proposer acceptable
        if (
            target_receiver not in receiver_ranks
            or proposer not in receiver_ranks[target_receiver]
        ):
            free_proposers.appendleft(proposer)
            continue

        current_match = receiver_matches.get(target_receiver)

        if current_match is None:
            # Case A: Receiver is free -> Accept & Hold
            receiver_matches[target_receiver] = proposer
        else:
            # Case B: Receiver evaluates current held match vs new proposal
            current_rank = receiver_ranks[target_receiver][current_match]
            new_rank = receiver_ranks[target_receiver][proposer]

            if new_rank < current_rank:
                # Receiver prefers the new proposer: switch hold and free former match
                receiver_matches[target_receiver] = proposer
                free_proposers.append(current_match)
            else:
                # Receiver rejects new proposer; proposer stays free
                free_proposers.appendleft(proposer)

    # 3. Format output as a list of (proposer, receiver) tuples, sorted by proposer name
    return sorted(
        [(proposer, receiver) for receiver, proposer in receiver_matches.items()]
    )

## Examples

In [15]:
# ==========================================
# Example Usage: Hospitals
# ==========================================
# Proposers (e.g., Doctors / Students)
proposers = {
    "Alice": ["Hospital_B", "Hospital_A", "Hospital_C"],
    "Bob":   ["Hospital_A", "Hospital_C", "Hospital_B"],
    "Carol": ["Hospital_A", "Hospital_B", "Hospital_C"]
}

# Receivers (e.g., Hospitals / Schools)
receivers = {
    "Hospital_A": ["Alice", "Bob", "Carol"],
    "Hospital_B": ["Bob", "Alice", "Carol"],
    "Hospital_C": ["Carol", "Alice", "Bob"]
}

matches = gale_roth_shapley(proposers, receivers)

print("Stable Matching Results (Receiver -> Proposer):")
for receiver, proposer in matches:
    print(f"  {receiver} <---> {proposer}")

Stable Matching Results (Receiver -> Proposer):
  Alice <---> Hospital_B
  Bob <---> Hospital_A
  Carol <---> Hospital_C


In [13]:
# ==========================================
# Example Usage: Mariage
# ==========================================
# Proposers (Men)
suitor_preferences = {
    "Adam": ["Diane", "Beth", "Carol", "Abby"],
    "Brad": ["Abby", "Beth", "Diane", "Carol"],
    "Cole": ["Beth", "Abby", "Carol", "Diane"],
    "Dan":  ["Abby", "Diane", "Carol", "Beth"]
}

# Receivers (Women)
reviewer_preferences = {
    "Abby":  ["Cole", "Adam", "Brad", "Dan"],
    "Beth":  ["Adam", "Cole", "Brad", "Dan"],
    "Carol": ["Dan", "Brad", "Adam", "Cole"],
    "Diane": ["Brad", "Adam", "Cole", "Dan"]
}

matches = gale_roth_shapley(suitor_preferences, reviewer_preferences)

for suitor, reviewer in matches:
    print(f"{suitor} is matched with {reviewer}")

Adam is matched with Diane
Brad is matched with Abby
Cole is matched with Beth
Dan is matched with Carol


### Example against an in-built Pyhton Function

In [16]:
!pip install matching -q
# ==========================================
# Example Usage: Mariage
# Example with a Prebuilt Python library
# ==========================================

from matching.games import StableMarriage

# Proposers (Men)
suitor_preferences = {
    "Adam": ["Diane", "Beth", "Carol", "Abby"],
    "Brad": ["Abby", "Beth", "Diane", "Carol"],
    "Cole": ["Beth", "Abby", "Carol", "Diane"],
    "Dan":  ["Abby", "Diane", "Carol", "Beth"]
}

# Receivers (Women)
reviewer_preferences = {
    "Abby":  ["Cole", "Adam", "Brad", "Dan"],
    "Beth":  ["Adam", "Cole", "Brad", "Dan"],
    "Carol": ["Dan", "Brad", "Adam", "Cole"],
    "Diane": ["Brad", "Adam", "Cole", "Dan"]
}

game = StableMarriage.create_from_dictionaries(suitor_preferences, reviewer_preferences)
matching = game.solve()

for suitor, reviewer in matching.items():
    print(f"{suitor} is matched with {reviewer}")

Adam is matched with Diane
Brad is matched with Abby
Cole is matched with Beth
Dan is matched with Carol


In [17]:
matching

{Adam: Diane, Brad: Abby, Cole: Beth, Dan: Carol}

## Unit tests

In [20]:
import unittest
from typing import List, Tuple, Dict


def is_stable(
    matches: List[Tuple[str, str]],
    proposers_pref: Dict[str, List[str]],
    receivers_pref: Dict[str, List[str]]
) -> bool:
    """
    Helper function to verify whether a matching has blocking pairs or blocking agents.
    """
    match_dict_p = dict(matches)
    match_dict_r = {r: p for p, r in matches}

    # 1. Individual rationality check: agents only matched to acceptable partners
    for p, r in matches:
        if r not in proposers_pref.get(p, []):
            return False
        if p not in receivers_pref.get(r, []):
            return False

    # 2. Blocking pair check:
    # A pair (p, r) blocks if p prefers r to their match, and r prefers p to their match
    for p, p_list in proposers_pref.items():
        p_current_match = match_dict_p.get(p)
        p_current_rank = p_list.index(p_current_match) if p_current_match in p_list else float('inf')

        for r in p_list:
            # Only consider receivers preferred over p's current match
            if p_list.index(r) >= p_current_rank:
                break

            r_list = receivers_pref.get(r, [])
            if p not in r_list:
                continue

            r_current_match = match_dict_r.get(r)
            r_current_rank = r_list.index(r_current_match) if r_current_match in r_list else float('inf')

            # If receiver also prefers p over their current partner, it's a blocking pair
            if r_list.index(p) < r_current_rank:
                return False

    return True


class TestGaleRothShapley(unittest.TestCase):

    def test_standard_marriage_case(self):
        """Tests standard 4x4 marriage instance from the earlier walkthrough."""
        suitors = {
            "Adam": ["Diane", "Beth", "Carol", "Abby"],
            "Brad": ["Abby", "Beth", "Diane", "Carol"],
            "Cole": ["Beth", "Abby", "Carol", "Diane"],
            "Dan":  ["Abby", "Diane", "Carol", "Beth"]
        }
        reviewers = {
            "Abby":  ["Cole", "Adam", "Brad", "Dan"],
            "Beth":  ["Adam", "Cole", "Brad", "Dan"],
            "Carol": ["Dan", "Brad", "Adam", "Cole"],
            "Diane": ["Brad", "Adam", "Cole", "Dan"]
        }
        expected = [
            ("Adam", "Diane"),
            ("Brad", "Abby"),
            ("Cole", "Beth"),
            ("Dan", "Carol")
        ]
        result = gale_roth_shapley(suitors, reviewers)
        self.assertEqual(result, expected)
        self.assertTrue(is_stable(result, suitors, reviewers))

    def test_proposer_optimality_and_asymmetry(self):
        """Tests that flipping sides produces the respective optimal stable matching."""
        m_prefs = {
            "M1": ["W1", "W2"],
            "M2": ["W2", "W1"]
        }
        w_prefs = {
            "W1": ["M2", "M1"],
            "W2": ["M1", "M2"]
        }

        # Men propose -> Men get their 1st choices
        men_propose = gale_roth_shapley(m_prefs, w_prefs)
        self.assertEqual(men_propose, [("M1", "W1"), ("M2", "W2")])
        self.assertTrue(is_stable(men_propose, m_prefs, w_prefs))

        # Women propose -> Women get their 1st choices
        women_propose = gale_roth_shapley(w_prefs, m_prefs)
        self.assertEqual(women_propose, [("W1", "M2"), ("W2", "M1")])
        self.assertTrue(is_stable(women_propose, w_prefs, m_prefs))

    def test_empty_inputs(self):
        """Tests behavior when market inputs are empty."""
        self.assertEqual(gale_roth_shapley({}, {}), [])

    def test_single_pair(self):
        """Tests minimal 1-to-1 market."""
        suitors = {"A": ["X"]}
        reviewers = {"X": ["A"]}
        self.assertEqual(gale_roth_shapley(suitors, reviewers), [("A", "X")])

    def test_unequal_market_sizes(self):
        """Tests market with more proposers than receivers (one remains unmatched)."""
        suitors = {
            "P1": ["R1"],
            "P2": ["R1"]
        }
        reviewers = {
            "R1": ["P2", "P1"]
        }
        # R1 prefers P2, leaving P1 unmatched
        result = gale_roth_shapley(suitors, reviewers)
        self.assertEqual(result, [("P2", "R1")])
        self.assertTrue(is_stable(result, suitors, reviewers))

    def test_unacceptable_partners(self):
        """Tests that agents are left unmatched if mutually acceptable terms aren't met."""
        suitors = {
            "P1": ["R1", "R2"],
            "P2": ["R2"]
        }
        reviewers = {
            # R1 does not list P1 (P1 is unacceptable to R1)
            "R1": ["P2"],
            "R2": ["P1", "P2"]
        }
        result = gale_roth_shapley(suitors, reviewers)

        # P1 proposes to R1 (rejected as unacceptable), then proposes to R2 (accepted)
        # P2 proposes to R2, but R2 prefers P1, so P2 is rejected and left unmatched
        self.assertEqual(result, [("P1", "R2")])
        self.assertTrue(is_stable(result, suitors, reviewers))

In [22]:
# Run cleanly inside Colab / IPython
unittest.main(argv=[''], verbosity=2, exit=False)

test_empty_inputs (__main__.TestGaleRothShapley.test_empty_inputs)
Tests behavior when market inputs are empty. ... ok
test_proposer_optimality_and_asymmetry (__main__.TestGaleRothShapley.test_proposer_optimality_and_asymmetry)
Tests that flipping sides produces the respective optimal stable matching. ... ok
test_single_pair (__main__.TestGaleRothShapley.test_single_pair)
Tests minimal 1-to-1 market. ... ok
test_standard_marriage_case (__main__.TestGaleRothShapley.test_standard_marriage_case)
Tests standard 4x4 marriage instance from the earlier walkthrough. ... ok
test_unacceptable_partners (__main__.TestGaleRothShapley.test_unacceptable_partners)
Tests that agents are left unmatched if mutually acceptable terms aren't met. ... ok
test_unequal_market_sizes (__main__.TestGaleRothShapley.test_unequal_market_sizes)
Tests market with more proposers than receivers (one remains unmatched). ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.009s

